# 关于“英雄武器”判定逻辑部分的探索与实现
## 逻辑探索
在最初的设想中，从经验上认为满足队伍当中只有1~2把长枪（高价值武器），其余队友均使用短枪的情况为“英雄武器”策略。
但由于数据来源层面的种种限制，无法精准获取每一回合每位选手使用的武器。

于是，将目光优先放在能获取的数据部分，尝试通过现有数据，反向推断出武器信息。最终收集到比赛回合的双方经济情况，回合胜负，击杀相关信息（包括使用武器）
在权衡可用数据量与分析可信度之后，最终采用了通过击杀信息中的武器使用情况，来近似真实武器使用情况的办法。即只研究采用“英雄武器”策略并且发生“英雄武器”击杀的回合。这也使得结论天然存在一定的选择性偏差。

## 代码实现
### 最终判定逻辑
通过调整相关参数与初步分析，最终确定下来的判断条件如下：
1. **武器相对昂贵**
武器价格 > 1.2倍该回合该队伍平均经济 
2. **击杀者队伍处于低经济状态**
经济类型属于 `Eco`、`Semi-Eco` 或 `Semi-Buy`。
3. **击杀者队伍相对对手处于经济劣势**
劣势方经济 < 0.75 *优势方经济
### 插曲——巧妙的对应关系
在这里处理过程中遇到了一个问题：判断过程需要进行武器价值与团队经济的比较，而原始数据中与击杀者武器相关联的信息只有击杀者的side（CT/T）和选手名称
（如:"killer": "TeSeS", "killer_side": "CT", "weapon": "USP Silenced"）
但团队经济储存的key为队伍名称，因此需要通过选手反推回所属队伍。（side相关数据实在难以得到相应队伍，因此放弃这种方法）初步解决方案采用了预设选手大名单的方法，即提前收集不同队伍选手情况，预设选手所在队伍。但这种处理方式无法应对选手转会前后的情况，会导致较大量数据的丢弃。

经过观察得到数据信息发现："economy"中"team-a:..., team-b:..."队伍前后顺序恰好是该局比赛"players"选手名单中选手出战顺序，即前五个选手属于team-a,后五个选手属于team-b，由此我们成功获取了对每局比赛动态建立选手到队伍的映射。具体代码实现如下：

In [ ]:
def process_match(match):
    for r in match['rounds']:
        eco = r['economy']
        r['economy'] = {team.lower(): info for team, info in eco.items()}
#标准化处理（方便后续对比）
        fk = r.get('first_kill')
        if fk is not None:
            fk['killer'] = fk['killer'].lower()
            fk['victim'] = fk['victim'].lower()

        other_kills = r.get('other_kills')
        if other_kills is not None:
            for k in other_kills:
                k['killer'] = k['killer'].lower()
                k['victim'] = k['victim'].lower()
#选取第一回合经济情况中两队伍名称作为标准
    teams = list(match['rounds'][0]['economy'].keys())
    team1, team2 = teams[0], teams[1]

    player_team = {}
    for i, (pid, name) in enumerate(match['players'].items()):
        player_team[name.lower()] = team1 if i < 5 else team2

    return {
        'players': player_team,
        'rounds': match['rounds']
    }

    with open('matches_raw.jsonl', 'r', encoding='utf-8') as f_in, \
         open('matches_normalized.jsonl', 'w', encoding='utf-8') as f_out:
        for line in f_in:
            match = json.loads(line)
            processed = process_match(match)
            f_out.write(json.dumps(processed, ensure_ascii=False) + '\n')

### 最终判定代码
进一步，通过预设武器价格列表，可实现对“英雄武器”的明确判定

In [ ]:
weapon_price = {
    '': 0,
    'AK-47': 2700,
    'AUG': 3300,
   # ......
}
def is_hero_kill_single(kill, round_economy, player_team):
    weapon = kill.get('weapon')
    if not weapon:
        return False
    
    weapon_value = weapon_price.get(weapon, 0)
    killer = kill.get('killer')
    victim = kill.get('victim')
    if not killer or not victim:
        return False
    
    killer_team = player_team.get(killer)
    victim_team = player_team.get(victim)
    if not killer_team or not victim_team:
        return False
    
    killer_team_value = round_economy.get(killer_team, {}).get('value', 0)
    victim_team_value = round_economy.get(victim_team, {}).get('value', 0)
    killer_team_ecotype = round_economy.get(killer_team, {}).get('type', '')
   #判断条件
    return ((weapon_value > killer_team_value * 0.2 *1.2) and             
            (killer_team_ecotype in ['Semi-Buy', 'Semi-Eco', 'Eco']) and  
            (killer_team_value < victim_team_value * 0.75))              
def add_is_hero_kill_to_match(match):
    player_team = match.get('player_team', {})
    for round_data in match.get('rounds', []):
        economy = round_data.get('economy', {})
        first = round_data.get('first_kill')
        if first and isinstance(first, dict):
            first['is_hero_kill'] = is_hero_kill_single(first, economy, player_team)
        for kill in round_data.get('other_kills', []):
            kill['is_hero_kill'] = is_hero_kill_single(kill, economy, player_team)
    return match

for idx in matches.index:
    rounds_list = matches.at[idx, 'rounds']
    match_dict = {'player_team': matches.at[idx, 'player_team'], 'rounds': rounds_list}
    add_is_hero_kill_to_match(match_dict)

# 关于首杀区域的探索与分析
根据组员初步分析得到：回合首杀与胜率之间有较强相关性。
研究命题：“英雄武器”策略下，发生在不同区域的首杀对当前回合胜率影响是否不同。

以Mirage地图为例，该地图注重中路的争夺，防守方一般需要较多的兵力安排在中路区域。意味着与中路连通性差的B区，如果被拿到首杀后可能要面临局部少打多的情况。初步猜测为B2，Vip这种位置的首杀可能对胜率的影响更强。
## 区域划分
### 从数字到区域——线性变换
利用大模型得到了一个建议html网站，通过手动标定一些点的坐标数据，计算得到其最小二乘估计，并返回变换矩阵，从而可通过数字坐标得地图的对应像素坐标。
根据游戏中常见定义，按照矩形划分了如下区域：

In [ ]:
def add_region(region):
    name = region["name"]
    (x1,y1),(x2,y2) = region["vertices"]
    regions.append((name, min(x1,x2), min(y1,y2), max(x1,x2), max(y1,y2)))
regions = []
add_region({"name": "A1", "vertices": [(490, 240), (380, 335)]})#A1
add_region({"name": "A2", "vertices": [(395, 336), (540, 420)]})#A2
#......
add_region({"name": "A_side", "vertices": [(379, 275), (215, 450)]})#A_side
add_region({"name": "Vip", "vertices": [(270, 170), (240, 274)]})#Vip
add_region({"name": "Connector", "vertices": [(285, 181), (340, 274)]})#Connector
add_region({"name": "T-mid", "vertices": [(400, 100), (470, 235)]})#T-mid
add_region({"name": "B1", "vertices": [(250, 70), (285, 169)]})#B1
add_region({"name": "B1", "vertices": [(286, 70), (350, 180)]})#B1
add_region({"name": "B2", "vertices": [(100, 0), (275, 40)]})#B2
add_region({"name": "B_side", "vertices": [(65, 35), (250, 150)]})#B_side

def point_in_rect(px, py, xmin, ymin, xmax, ymax):
    return xmin <= px <= xmax and ymin <= py <= ymax

def get_region(x, y):
    for name, xmin, ymin, xmax, ymax in regions:
        if point_in_rect(x, y, xmin, ymin, xmax, ymax):
            return name
    return "Elsewhere"

由此对所有坐标得到区域分组，利用get_region函数，可以实现一些基础统计图表按区域分组后的呈现。
### 分析区域变量“净”效益——二元逻辑回归
在前分析中初步证实了首杀对回合胜率和经济对胜率的影响，尝试通过二元逻辑回归的方式计算

### 1. 数据收集：`collect_regression_with_position`

该函数遍历 `matches_disadvantage` 数据集，为每个经济劣势回合提取一条样本记录。

#### 1.1 输入与输出
- **输入**：`matches_df`，结构为每个比赛包含 `player_team`（选手到队伍的映射）和 `rounds` 列表。
- **输出**：`pd.DataFrame`，包含以下字段：
  - `disadvantage_won`：劣势方是否获胜（0/1）
  - `is_hero_round`：是否英雄回合（0/1）
  - `disadvantage_team_value`：劣势方本回合经济总值
  - `disadvantage_side`：劣势方阵营（`CT`/`T`）
  - `first_kill_position`：首杀位置（若无首杀或优势方首杀，则为 `"NoFirstKill"`；否则由 `get_region` 映射为区域名）

#### 1.2 关键步骤解析
比较两支队伍的经济值，将总值较小的一方标记为劣势方。

In [ ]:
if val1 < val2:
    dis_team = teams[0]
    dis_value = val1
else:
    dis_team = teams[1]
    dis_value = val2

判断首杀归属与位置

In [ ]:
first = round_data.get('first_kill')
if first:
    killer = first.get('killer')
    if killer and player_team.get(killer) == dis_team:
        # 劣势方取得首杀 → 获取坐标并映射区域
        pos_str = first.get('killer_pos')
        # 此处省略将坐标转化为数字对过程
        region = get_region(x, y)
    else:
        region = 'NoFirstKill'  # 优势方首杀或无首杀信息
else:
    region = 'NoFirstKill'

利用 get_team_side_map 确定劣势方的阵营（CT/T），再与 winner_side 比较得到是否获胜。

In [ ]:
team_side = get_team_side_map(round_data, player_team)
dis_side = team_side.get(dis_team)
winner_side = round_data.get('winner_side')
dis_won = (dis_side == winner_side) if winner_side else False

### 2. 特征工程

In [ ]:
reg_pos_df['disadvantage_team_value_scaled'] = reg_pos_df['disadvantage_team_value'] / 1000
reg_pos_df['disadvantage_side_encoded'] = (reg_pos_df['disadvantage_side'] == 'T').astype(int)
pos_dummies = pd.get_dummies(reg_pos_df['first_kill_position'], prefix='pos', drop_first=True)

#### 2.1 经济缩放
将劣势方经济值除以 1000，使系数解释为“每增加 1000 美元”的影响，避免数值过大或过小导致的数值不稳定。

#### 2.2 阵营编码
disadvantage_side_encoded：T 方编码为 1，CT 方编码为 0。

#### 2.3 首杀位置虚拟变量
pd.get_dummies(..., drop_first=True)：将 first_kill_position 转换为多个 0/1 变量，并删除第一个类别（'A1' 或按字母排序的第一个）作为参照组。

注：代码中未显式删除参照类，但 drop_first=True 会自动避免多重共线性。

### 3. 模型构建与拟合

In [ ]:
X = pd.concat([reg_pos_df[['is_hero_round', 'disadvantage_team_value_scaled', 'disadvantage_side_encoded']], pos_dummies], axis=1)
X = sm.add_constant(X)   # 添加截距项
y = reg_pos_df['disadvantage_won']
model = sm.Logit(y, X).fit(maxiter=1000)

迭代算法
sm.Logit 使用迭代加权最小二乘法（IRLS），最大迭代次数设为 1000 以确保收敛。

### 4. 结果输出与解读
#### 4.1 模型摘要

输出示例：

                           Logit Regression Results
Dep. Variable:       disadvantage_won   No. Observations:                 6016
Model:                          Logit   Df Residuals:                     6011
Method:                           MLE   Df Model:                            4
Pseudo R-squ.:                  0.2027   Log-Likelihood:                -2648.8

                 coef    std err      z      P>|z|   [0.025   0.975]
const         -2.2656      0.110  -20.573   0.000   -2.481   -2.050
is_hero_round  1.1381      0.086   13.167   0.000    0.969    1.307
...
关键指标解释：

指标	                                            含义
coef	                                 对数几率（log-odds）的变化量
std err	                                      系数的标准误
z	                                    Wald 统计量 = coef / std err
P>|z|	                                  双尾 p 值，检验系数是否为 0
[0.025, 0.975]	                             系数的 95% 置信区间
Pseudo R-squ.	                    伪 R²（如 McFadden R²），衡量模型拟合优度
Log-Likelihood	                           对数似然值，用于模型比较

#### 4.2 优势比（Odds Ratio）

In [ ]:
odds_ratios = np.exp(model.params)
conf = np.exp(model.conf_int())
odds_df = pd.DataFrame({'Odds Ratio': odds_ratios, '2.5%': conf[0], '97.5%': conf[1]})

优势比 = e^{coef}：表示自变量每增加一个单位，优势（获胜概率与失败概率之比）的倍数变化。

OR > 1：该因素促进获胜。

OR < 1：该因素抑制获胜。

置信区间不包含 1 时具有统计显著性。

#### 4.3 首杀位置整体显著性检验（似然比检验）

In [ ]:
X_reduced = sm.add_constant(reg_pos_df[['is_hero_round', 'disadvantage_team_value_scaled', 'disadvantage_side_encoded']])
model_reduced = sm.Logit(y, X_reduced).fit(disp=0)
lr_stat = -2 * (model_reduced.llf - model.llf)
df = len(pos_dummies.columns)
p_val = 1 - stats.chi2.cdf(lr_stat, df)

原假设 H₀：所有首杀位置虚拟变量的系数同时为零（即位置不影响劣势方胜率）。
若 p 值 < 0.05，拒绝原假设，认为首杀位置整体显著。

本例中 p = 0.286，说明首杀位置无显著影响。
“英雄武器”回合首杀在哪并不重要，重要的是如何取得首杀，以及后续的收益维持转换率。